# Sub-1B document-VLM comparison on a T4 GPU

Thin runner: **clone → install → run repo scripts**. Runs every sub-1B model + **PaddleOCR-VL 1.0/1.5/1.6** on the capability, spatial/context, and **proposed custom-eval** sets, measuring **score + inference time + CPU/GPU memory**.

Runtime → Change runtime type → **T4 GPU**, then Run all.


## 1. GPU check

In [1]:
!nvidia-smi -L

GPU 0: Tesla T4 (UUID: GPU-8cba8321-ee75-5e94-3923-c41bf91987be)


## 2. Clone repo + install

In [2]:
%cd /content
![ -d OCR ] || git clone https://github.com/SangbumChoi/OCR.git
%cd /content/OCR
!git checkout claude/new-session-w79q0i && git pull --ff-only
!pip -q install -e '.[models,finetune]' protobuf

/content
Cloning into 'OCR'...
remote: Enumerating objects: 1846, done.
remote: Counting objects: 100% (337/337), done.
remote: Compressing objects: 100% (222/222), done.
remote: Total 1846 (delta 129), reused 269 (delta 93), pack-reused 1509 (from 1)
Receiving objects: 100% (1846/1846), 64.37 MiB | 17.32 MiB/s, done.
Resolving deltas: 100% (710/710), done.
/content/OCR
Branch 'claude/new-session-w79q0i' set up to track remote branch 'claude/new-session-w79q0i' from 'origin'.
Switched to a new branch 'claude/new-session-w79q0i'
Already up to date.
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 118.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 48.5 MB/s eta 0:00:

## 3. Run the full comparison (chat VLMs @ tf4.49, PaddleOCR-VL @ tf4.57; all measured)
Installs CJK fonts + QR/barcode libs, builds the probes incl. the custom-eval set, runs all models on capability + spatial/context + custom-eval, and aggregates.

In [3]:
!DEVICE=cuda bash scripts/run_full_comparison.sh

스트리밍 출력 내용이 길어서 마지막 5000줄이 삭제되었습니다.
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
A new version of the following files was downloaded from https://huggingface.co/OpenGVLab/InternVL3-1B:
- configuration_internvl_chat.py
- configuration_intern_vit.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
modeling_internvl_chat.py: 16.0kB [00:00, 36.5MB/s]
conversation.py: 15.3kB [00:00, 52.2MB/s]
A new version of the following files was downloaded from https://huggingface.co/OpenGVLab/InternVL3-1B:
- conversation.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
modeling_intern_vit.py: 18.1kB [00:00, 50.9MB/s]
A new version of the following files was downloaded from https://huggingface.co/Op

## 4. Scores + efficiency (time & memory)

In [4]:
print(open('docs/results/matrix_capability.md').read())

# Cross-benchmark result matrix (preview set)

Models run: 19/19 · benchmarks: 6

Per-cell = task score (ANLS / relaxed-acc / OCRBench / exact). One preview sample per benchmark, so treat as a *plumbing + sanity* matrix, not leaderboard accuracy.

| model            | cap_text | cap_kie | cap_integ_sum | cap_integ_rel | cap_chart | cap_ground |
| ---------------- | -------- | ------- | ------------- | ------------- | --------- | ---------- |
| dummy-echo       | 0.00     | 0.00    | 0.00          | 0.00          | 0.00      | 0.00       |
| florence2-base   | 0.00     | 0.00    | 0.00          | 0.00          | 0.00      | 0.00       |
| florence2-large  | 0.00     | 0.00    | 0.00          | 0.00          | 1.00      | 0.00       |
| got-ocr2         | 0.00     | 0.00    | 0.00          | 0.00          | 1.00      | 0.00       |
| h2ovl-0.8b       | 0.00     | 0.00    | 0.00          | 0.00          | 0.00      | 0.00       |
| internvl2-1b     | 0.93     | 1.00    | 1.00          | 0

## 5. Spatial / context shortcut-robust signals

In [5]:
print(open('docs/results/matrix_probe.md').read())
!python scripts/analyze_probe_signals.py --probe probe

# Cross-benchmark result matrix (preview set)

Models run: 19/19 · benchmarks: 15

Per-cell = task score (ANLS / relaxed-acc / OCRBench / exact). One preview sample per benchmark, so treat as a *plumbing + sanity* matrix, not leaderboard accuracy.

| model            | sp_quad_top-left | sp_quad_top-right | sp_quad_bottom-left | sp_quad_bottom-right | sp_relpos_normal | sp_relpos_counterfactual | sp_box_top | sp_box_mid | sp_box_bot | ctx_consistency_consistent | ctx_consistency_inconsistent | ctx_absence | ctx_distractor | ctx_xref_bob | ctx_xref_alice |
| ---------------- | ---------------- | ----------------- | ------------------- | -------------------- | ---------------- | ------------------------ | ---------- | ---------- | ---------- | -------------------------- | ---------------------------- | ----------- | -------------- | ------------ | -------------- |
| dummy-echo       | 0.00             | 0.00              | 0.00                | 0.00                 | 0.00             | 0

## 6. Proposed custom-eval — by class / language / rotation / direction / spotting

In [6]:
print(open('docs/results/custom_eval_breakdown.md').read())

# Custom-eval breakdown (proposed format)

Per-model scores sliced by the axes the format is built around.

## By content class

| model            | barcode | chart | direction | formula | logo  | orientation | qr  | stamp | table | text  |
| ---------------- | ------- | ----- | --------- | ------- | ----- | ----------- | --- | ----- | ----- | ----- |
| dummy-echo       | 0.0     | 0.0   | 0.0       | 0.0     | 0.0   | 0.0         | 0.0 | 0.0   | 0.0   | 0.012 |
| florence2-base   | 1.0     | 0.0   | 0.0       | 0.889   | 0.0   | 0.0         | 0.0 | 0.0   | 0.0   | 0.206 |
| florence2-large  | 1.0     | 1.0   | 0.0       | 0.889   | 0.0   | 0.0         | 0.0 | 0.0   | 0.0   | 0.23  |
| got-ocr2         | 1.0     | 1.0   | 0.0       | 0.625   | 0.0   | 0.0         | 0.0 | 0.0   | 0.0   | 0.379 |
| h2ovl-0.8b       | 0.0     | 0.0   | 0.0       | 0.0     | 0.0   | 0.0         | 0.0 | 0.0   | 0.0   | 0.0   |
| internvl2-1b     | 1.0     | 1.0   | 0.333     | 0.051   | 0.0   | 0.0        

## 7. PaddleOCR-VL 1.0 vs 1.5 vs 1.6

In [7]:
!for m in paddleocr-vl paddleocr-vl-1.5 paddleocr-vl-1.6; do echo "== $m =="; python3 -c "import json;s=json.load(open(f'docs/results/$m/custom_eval/summary.json'));print({k:s.get(k) for k in ['score','avg_latency_s','peak_gpu_mb']})" 2>/dev/null || echo 'n/a'; done

== paddleocr-vl ==
{'score': 0.2097, 'avg_latency_s': 5.491, 'peak_gpu_mb': 2116.7}
== paddleocr-vl-1.5 ==
{'score': 0.2652, 'avg_latency_s': 6.304, 'peak_gpu_mb': 2116.9}
== paddleocr-vl-1.6 ==
{'score': 0.2652, 'avg_latency_s': 5.413, 'peak_gpu_mb': 2116.7}


## 8. Download all results

In [8]:
!zip -qr /content/docvlm_results.zip results
from google.colab import files; files.download('/content/docvlm_results.zip')


zip error: Nothing to do! (try: zip -qr /content/docvlm_results.zip . -i results)


FileNotFoundError: Cannot find file: /content/docvlm_results.zip